In [ ]:
#Imports:

from pathlib import Path
import sys
import pandas as pd
import numpy as np
import urllib.request
import zipfile
import os

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import DATASET_URL, NASA_DATA_DIR, PROCESSED_DATA_DIR, TRAIN_FILE, PROCESSED_FILE_NAME

In [ ]:
#1. Configura??o do diret?rio:

data_path = NASA_DATA_DIR
processed_data_path = PROCESSED_DATA_DIR
os.makedirs(data_path, exist_ok=True)
os.makedirs(processed_data_path, exist_ok=True)

In [ ]:
#2. Download do Dataset:

print("Baixando dataset...")

url = DATASET_URL
zip_path = os.path.join(data_path, "CMAPSSData.zip")

In [ ]:
#3.Definindo o nome das colunas:

col_names = ['engine_id', 'time_cycle', 'op_setting_1', 'op_setting_2', 'op_setting_3']
sensor_cols = [f'sensor_{i}' for i in range(1, 22)]
col_names.extend(sensor_cols)

In [ ]:
#4. ETL:

train_file = os.path.join(data_path, TRAIN_FILE)

try:
    df_train = pd.read_csv(train_file, sep='\s+', header=None, names=col_names)
    print(f"Dados carregados: {df_train.shape[0]} linhas e {df_train.shape[1]} colunas.")
    
    # Descobre o ultimo ciclo de cada motor
    max_cycles = df_train.groupby('engine_id')['time_cycle'].max().reset_index()
    max_cycles.columns = ['engine_id', 'max_cycle']
    
    # Junta com o dataset original
    df_train = df_train.merge(max_cycles, on=['engine_id'], how='left')
    
    # Calcula o RUL: ciclo maximo do motor menos ciclo atual
    df_train['RUL'] = df_train['max_cycle'] - df_train['time_cycle']
    
    # Remove a coluna auxiliar
    df_train.drop('max_cycle', axis=1, inplace=True)
    
    print("
Primeiras linhas com a nova coluna RUL:")
    display(df_train[['engine_id', 'time_cycle', 'sensor_2', 'sensor_3', 'RUL']].head())
    
    # Salva o dado processado
    output_path = os.path.join(processed_data_path, PROCESSED_FILE_NAME)
    df_train.to_csv(output_path, index=False)
    print(f"
Dataset processado salvo com sucesso em {output_path}")

except FileNotFoundError:
    print(f"Baixar o arquivo CMAPSSData.zip, extrair e colocar o '{TRAIN_FILE}' na pasta '{data_path}'")